# Topic 20e - Reshaping with `melt` and `pivot`

Practice for section 6 of the [main lesson](topic20_data_wrangling.ipynb).

The same information can sit in two shapes:

- **Long**: one row per observation, a column naming the variable. Machines prefer this.
- **Wide**: one variable spread across the columns. People prefer this.

**Analyse long. Present wide.** `melt()` goes one way, `pivot_table()` the other.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_data_dir():
    """Locate the lesson's data folder from any sensible working directory."""
    for candidate in [Path.cwd() / "data",
                      Path.cwd() / "data_wrangling" / "data",
                      Path.cwd().parent / "data"]:
        if (candidate / "orders.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find orders.csv - check your data folder.")


DATA_DIR = find_data_dir()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 130)
plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["figure.dpi"] = 110

MONTH_ORDER = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def load_analysis_table():
    """The cleaned, validated, joined table built in the main lesson."""
    orders = pd.read_csv(DATA_DIR / "orders.csv", parse_dates=["order_date"])
    customers = pd.read_csv(DATA_DIR / "customers.csv")
    products = pd.read_csv(DATA_DIR / "products.csv")
    regions = pd.read_csv(DATA_DIR / "regions.csv")

    customers = customers.copy()
    customers["region"] = (customers["region"].str.strip()
                                              .str.replace(r"\s+", " ", regex=True)
                                              .str.title())
    customers = customers.merge(regions, on="region", how="left", validate="m:1")
    products = products.drop_duplicates(subset="product_id", keep="first")

    table = (orders
             .merge(customers, on="customer_id", how="left", validate="m:1")
             .merge(products, on="product_id", how="left", validate="m:1"))
    table["revenue_ghs"] = (table["quantity"] * table["unit_price_ghs"]).round(2)
    table["cost_ghs"] = (table["quantity"] * table["unit_cost_ghs"]).round(2)
    table["profit_ghs"] = (table["revenue_ghs"] - table["cost_ghs"]).round(2)
    table["month"] = table["order_date"].dt.strftime("%b")
    return table


analysis = load_analysis_table()
print(f"analysis table: {analysis.shape[0]} rows x {analysis.shape[1]} columns")

## 1. `melt()` - wide to long

Our targets file stores one column per month, which is unusable until melted.

In [ ]:
targets_wide = pd.read_csv(DATA_DIR / "monthly_targets_wide.csv")
print("As stored - 12 month columns:")
print(targets_wide.head(3).to_string(index=False))
print(f"\nShape: {targets_wide.shape}")

In [ ]:
targets = targets_wide.melt(id_vars="region",
                            var_name="month",
                            value_name="target_ghs")
print(f"{targets_wide.shape} melted into {targets.shape}  "
      f"(16 regions x 12 months = {16 * 12})")
print(targets.head(4).to_string(index=False))

The three arguments:

- `id_vars` - columns to **keep as they are**. Everything else gets melted.
- `var_name` - the new column that receives the old **headers**.
- `value_name` - the new column that receives the old **cell values**.

Only now can the targets be merged onto anything, because `month` is finally a variable rather than a column name.

## 2. `pivot_table()` - long to wide

Going the other way, aggregating as it goes.

In [ ]:
revenue_matrix = analysis.pivot_table(index="region", columns="month",
                                      values="revenue_ghs", aggfunc="sum",
                                      fill_value=0)
print("Columns as pivot_table returns them:")
print(list(revenue_matrix.columns))
print("\nAlphabetical, not calendar. Always reorder:")
revenue_matrix = revenue_matrix[MONTH_ORDER]
print(revenue_matrix.round(0).head(5).to_string())

## 3. `pivot()` versus `pivot_table()`

| | `pivot()` | `pivot_table()` |
|---|---|---|
| Aggregates | No | Yes, via `aggfunc` |
| Duplicate index/column pairs | Raises | Combines them |
| Use for | Data already one row per cell | Summarising raw records |

In [ ]:
try:
    analysis.pivot(index="region", columns="month", values="revenue_ghs")
except ValueError as error:
    print("pivot() refused:")
    print(" ", str(error)[:100], "...")
    print("\nCorrect: there are many orders per region per month, so there is")
    print("no single value for each cell. pivot_table(aggfunc='sum') is the")
    print("right tool because it says what to do with the many values.")

## 4. Several aggregations at once

`aggfunc` accepts a list, or a dictionary mapping columns to functions.

In [ ]:
summary = analysis.pivot_table(
    index="zone",
    values=["revenue_ghs", "profit_ghs", "quantity"],
    aggfunc={"revenue_ghs": "sum", "profit_ghs": "sum", "quantity": "mean"},
).round(2)
print(summary.to_string())

print("\nWith margins=True, pandas adds an 'All' total row:")
totals = analysis.pivot_table(index="zone", columns="channel",
                              values="revenue_ghs", aggfunc="sum",
                              fill_value=0, margins=True).round(0)
print(totals.to_string())

## 5. Reshaping makes a join possible

This is the payoff. The targets could not be merged while they were wide.

In [ ]:
actuals = (analysis.groupby(["region", "month"], as_index=False)["revenue_ghs"]
                   .sum()
                   .rename(columns={"revenue_ghs": "actual_ghs"}))

performance = actuals.merge(targets, on=["region", "month"],
                            how="left", validate="1:1")
performance["achieved_pct"] = (performance["actual_ghs"]
                               / performance["target_ghs"] * 100).round(1)

print(f"actuals {actuals.shape} + targets {targets.shape} -> {performance.shape}")
print("\nFive worst region-months against target:")
print(performance.nsmallest(5, "achieved_pct").to_string(index=False))

## 6. `stack()` and `unstack()`

The same conversion applied to the **index** instead of the columns. Most often used to flatten a two-key `groupby`.

In [ ]:
grouped = analysis.groupby(["zone", "channel"])["revenue_ghs"].sum()
print("Two index levels (long):")
print(grouped.head(4).to_string())
print("\n.unstack() - inner level becomes columns (wide):")
print(grouped.unstack().round(0).to_string())

## 7. Practice

1. Build a wide table of **order counts** by `category` (rows) and `channel` (columns), using `aggfunc="size"` and `fill_value=0`.
2. Which category-channel combination has the most orders?
3. Melt your wide table back to long, with columns `category`, `channel`, `order_count`.
4. Confirm the grand total is identical in both shapes.
5. Build a wide table of **mean profit** by `zone` and `size_band`, then say in one sentence what it shows.

In [ ]:
# YOUR CODE HERE

**Your answer to question 5:**

*(Write your answer here.)*

---
## Where this fits

- [Back to the main lesson](topic20_data_wrangling.ipynb)
- Previous: [20d - Merge hazards and validation](topic20d_merge_hazards_and_validation.ipynb)
- Next: [20f - Transforms and binning](topic20f_transforms_and_binning.ipynb)